In [ ]:
import cutlass
import cutlass.cute as cute


## Tensor

A tensor in CuTe is created through the composition of two key components:

1. An **Engine** (E) - A random-access, pointer-like object that supports:
   - Offset operation: `e + d → e` (offset engine by elements of a layout's codomain)
   - Dereference operation: `*e → v` (dereference engine to produce value)

2. A **Layout** (L) - Defines the mapping from coordinates to offsets

A tensor is formally defined as the composition of an engine E with a layout L, expressed as `T = E ∘ L`. When evaluating a tensor at coordinate c, it:

1. Maps the coordinate c to the codomain using the layout
2. Offsets the engine accordingly
3. Dereferences the result to obtain the tensor's value

This can be expressed mathematically as:

```
T(c) = (E ∘ L)(c) = *(E + L(c))
```

## Example Usage

Here's a simple example of creating a tensor using pointer and layout `(8,5):(5,1)` and fill with ones:
## 张量 (Tensor)
在 CuTe 中，张量的创建是通过两个核心组件的组合实现的：

1. 存储引擎 (Engine, E) — 一个支持随机访问、类似指针的对象，它支持以下操作：
    偏移操作：e + d → e（根据布局值域中的元素对引擎进行偏移）
    解引用操作：*e → v（对引擎进行解引用以产生数值）

2. 布局 (Layout, L) — 定义了从逻辑坐标到物理偏移量的映射关系。

张量在形式上被定义为引擎 $E$ 与布局 $L$ 的复合，表示为 $T = E \circ L$。当在坐标 $c$ 处对张量求值时，它会执行以下步骤：
1. 使用布局将坐标 $c$ 映射到其值域（即计算偏移量）。

2. 相应地偏移引擎。

3. 对结果进行解引用以获取张量的数值。

这一过程可以用数学公式表示为：

$$T(c) = (E \circ L)(c) = *(E + L(c))$$

In [ ]:
@cute.jit
def create_tensor_from_ptr(ptr: cute.Pointer):
    layout = cute.make_layout((8, 5), stride=(5, 1))
    tensor = cute.make_tensor(ptr, layout)
    tensor.fill(1)
    cute.print_tensor(tensor)

This creates a tensor where:
- The engine is a pointer
- The layout with shape `(8, 5)` and stride `(5, 1)`
- The resulting tensor can be evaluated using coordinates defined by the layout

We can test this by allocating buffer with torch and run test with pointer to torch tensor

In [ ]:
import torch

from cutlass.torch import dtype as torch_dtype
import cutlass.cute.runtime as cute_rt

a = torch.randn(8, 5, dtype=torch_dtype(cutlass.Float32))
print(a)
ptr_a = cute_rt.make_ptr(cutlass.Float32, a.data_ptr())

create_tensor_from_ptr(ptr_a)

In [ ]:
tensor([[ 0.4663,  0.6554, -1.1983, -0.9660,  0.5563],
        [-0.4078,  1.0326, -0.6022, -0.2086,  0.2869],
        [-0.3074,  0.2124, -0.3802,  1.0927, -0.1382],
        [-0.4967,  0.4220,  0.2201, -0.4163,  1.8213],
        [-0.6584,  1.7414,  0.1889,  0.2382, -1.4296],
        [-2.2987, -0.6195,  0.3881, -1.0199, -0.7136],
        [-0.9205,  0.5304,  0.7026, -1.0266, -0.4718],
        [-0.7185, -1.0821, -0.5227, -0.8727, -0.0373]])
tensor(raw_ptr(0x000000002c8efbc0: f32, generic, align<4>) o (8,5):(5,1), data=
       [[ 1.000000,  1.000000,  1.000000,  1.000000,  1.000000, ],
        [ 1.000000,  1.000000,  1.000000,  1.000000,  1.000000, ],
        [ 1.000000,  1.000000,  1.000000,  1.000000,  1.000000, ],
        ...
        [ 1.000000,  1.000000,  1.000000,  1.000000,  1.000000, ],
        [ 1.000000,  1.000000,  1.000000,  1.000000,  1.000000, ],
        [ 1.000000,  1.000000,  1.000000,  1.000000,  1.000000, ]])

## DLPACK support 

CuTe DSL is designed to support dlpack protocol natively. This offers easy integration with frameworks 
supporting DLPack, e.g. torch, numpy, jax, tensorflow, etc.

For more information, please refer to DLPACK project: https://github.com/dmlc/dlpack

Calling `from_dlpack` can convert any tensor or ndarray object supporting `__dlpack__` and `__dlpack_device__`.

In [ ]:
from cutlass.cute.runtime import from_dlpack


@cute.jit
def print_tensor_dlpack(src: cute.Tensor):
    print(src)
    cute.print_tensor(src)

In [ ]:
a = torch.randn(8, 5, dtype=torch_dtype(cutlass.Float32))

print_tensor_dlpack(from_dlpack(a))

In [ ]:
tensor<ptr<f32, generic> o (8,5):(5,1)>
tensor(raw_ptr(0x000000002ca3ac00: f32, generic, align<4>) o (8,5):(5,1), data=
       [[-0.300179, -0.329489,  0.232334, -0.826196, -0.234135, ],
        [ 0.041298,  1.325581, -1.055890,  0.780220,  0.067543, ],
        [-1.507660, -0.419070,  0.629296,  0.513014, -1.548166, ],
        ...
        [-0.739360, -1.351777,  0.275485, -0.162179,  1.357413, ],
        [ 0.829721,  0.202159,  0.362681, -0.182584,  1.870800, ],
        [ 0.601124, -0.050782,  0.822122,  0.803683, -0.075312, ]])


In [ ]:
import numpy as np

a = np.random.randn(8, 8).astype(np.float32)

print_tensor_dlpack(from_dlpack(a))

In [ ]:
tensor<ptr<f32, generic> o (8,8):(8,1)>
tensor(raw_ptr(0x000000002ca49dc0: f32, generic, align<4>) o (8,8):(8,1), data=
       [[ 1.679403,  2.421720,  0.046635, ..., -0.762795,  0.269486,  0.799897, ],
        [-0.238368, -1.417304,  0.830548, ...,  0.231837, -0.136711, -0.069025, ],
        [ 0.629811,  1.272897,  0.246972, ...,  1.567207, -0.183601, -1.217345, ],
        ...
        [-0.526581, -1.115891, -0.110309, ...,  0.660223, -0.305523, -1.076966, ],
        [-0.769984, -0.937175, -0.285849, ...,  1.508988,  0.016382,  0.665918, ],
        [ 0.313322,  0.628846, -0.827665, ..., -0.102310, -0.114555, -1.475539, ]])

## Tensor Evaluation Methods

Tensors support two primary methods of evaluation:

### 1. Full Evaluation
When applying the tensor evaluation with a complete coordinate c, it computes the offset, applies it to the engine, 
and dereferences it to return the stored value. This is the straightforward case where you want to access 
a specific element of the tensor.

### 2. Partial Evaluation (Slicing)
When evaluating with an incomplete coordinate c = c' ⊕ c* (where c* represents the unspecified portion), 
the result is a new tensor which is a slice of the original tensor with its engine offset to account for 
the coordinates that were provided. This operation can be expressed as:

```
T(c) = (E ∘ L)(c) = (E + L(c')) ∘ L(c*) = T'(c*)
```

Slicing effectively reduces the dimensionality of the tensor, creating a sub-tensor that can be 
further evaluated or manipulated.

## 张量求值方法

张量支持两种主要的求值方式：

1. 全求值 (Full Evaluation)当使用完整的坐标 $c$ 对张量进行求值时，它会计算出偏移量（Offset），将其应用于存储引擎（Engine），并进行解引用（Dereference）以返回存储的数值。这是想要访问张量中特定元素的直接情况。

2. 部分求值 / 切片 (Partial Evaluation / Slicing)当使用不完整的坐标 $c = c' \oplus c^*$ 进行求值时（其中 $c^*$ 代表未指定的部分），其结果是一个新张量。该新张量是原始张量的切片，其引擎地址已根据所提供的坐标 $c'$ 进行了偏移补偿。

该操作可以表示为：
$$T(c) = (E \circ L)(c) = (E + L(c')) \circ L(c^*) = T'(c^*)$$

切片操作有效地降低了张量的维度，创建了一个可以进一步求值或操作的子张量（Sub-tensor）。

In [ ]:
@cute.jit
def tensor_access_item(a: cute.Tensor):
    # access data using linear index
    cute.printf(
        "a[2] = {} (equivalent to a[{}])",
        a[2],
        cute.make_identity_tensor(a.layout.shape)[2],
    )
    cute.printf(
        "a[0] = {} (equivalent to a[{}])",
        a[0],
        cute.make_identity_tensor(a.layout.shape)[0],
    )
    cute.printf(
        "a[9] = {} (equivalent to a[{}])",
        a[9],
        cute.make_identity_tensor(a.layout.shape)[9],
    )

    # access data using n-d coordinates, following two are equivalent
    cute.printf("a[2,0] = {}", a[2, 0])
    cute.printf("a[2,4] = {}", a[2, 4])
    cute.printf("a[(2,4)] = {}", a[2, 4])

    # assign value to tensor@(2,4)
    a[2, 3] = 100.0
    a[2, 4] = 101.0
    cute.printf("a[2,3] = {}", a[2, 3])
    cute.printf("a[(2,4)] = {}", a[(2, 4)])


# Create a tensor with sequential data using torch
data = torch.arange(0, 8 * 5, dtype=torch.float32).reshape(8, 5)
tensor_access_item(from_dlpack(data))

print(data)

In [ ]:
a[2] = 10.000000 (equivalent to a[(2,0)])
a[2] = 0.000000 (equivalent to a[(0,0)])
a[9] = 6.000000 (equivalent to a[(1,1)])
a[2,0] = 10.000000
a[2,4] = 14.000000
a[(2,4)] = 14.000000
a[2,3] = 100.000000
a[(2,4)] = 101.000000
tensor([[  0.,   1.,   2.,   3.,   4.],
        [  5.,   6.,   7.,   8.,   9.],
        [ 10.,  11.,  12., 100., 101.],
        [ 15.,  16.,  17.,  18.,  19.],
        [ 20.,  21.,  22.,  23.,  24.],
        [ 25.,  26.,  27.,  28.,  29.],
        [ 30.,  31.,  32.,  33.,  34.],
        [ 35.,  36.,  37.,  38.,  39.]])

### Tensor as memory view

In CUDA programming, different memory spaces have different characteristics in terms of access speed, scope, and lifetime:

- **generic**: Default memory space that can refer to any other memory space.
- **global memory (gmem)**: Accessible by all threads across all blocks, but has higher latency.
- **shared memory (smem)**: Accessible by all threads within a block, with much lower latency than global memory.
- **register memory (rmem)**: Thread-private memory with the lowest latency, but limited capacity.
- **tensor memory (tmem)**: Specialized memory introduced in NVIDIA Blackwell architecture for tensor operations.

When creating tensors in CuTe, you can specify the memory space to optimize performance based on your access patterns.

For more information on CUDA memory spaces, see the [CUDA Programming Guide](https://docs.nvidia.com/cuda/cuda-c-programming-guide/index.html#memory-hierarchy).


### 张量作为内存视图

在 CUDA 编程中，不同的内存空间在访问速度、范围和生命周期方面具有不同的特性：

- **generic**：默认内存空间，可以引用任何其他内存空间。
- **全局内存 (gmem)**：所有块中的所有线程都可以访问，但具有更高的延迟。
- **共享内存 (smem)**：块内的所有线程都可以访问，与全局内存相比具有更低的延迟。
- **寄存器内存 (rmem)**：线程私有内存，具有最低的延迟，但容量有限。
- **张量内存 (tmem)**：NVIDIA Blackwell 架构中引入的专门用于张量操作的内存。

在 CuTe 中创建张量时，您可以指定内存空间以基于您的访问模式优化性能。

有关 CUDA 内存空间的更多信息，请参阅 [CUDA 编程指南](https://docs.nvidia.com/cuda/cuda-c-programming-guide/index.html#memory-hierarchy)。

## Coordinate Tensors

### Definition and Properties

A coordinate tensor $T: Z^n → Z^m$ is a mathematical structure that establishes a mapping between coordinate spaces. Unlike standard tensors that map coordinates to scalar values, coordinate tensors map coordinates to other coordinates, forming a fundamental building block for tensor operations and transformations.

### Examples

Consider a `(4,4)` coordinate tensor:

**Row-Major Layout (C-style):**
\begin{bmatrix} 
(0,0) & (0,1) & (0,2) & (0,3) \\
(1,0) & (1,1) & (1,2) & (1,3) \\
(2,0) & (2,1) & (2,2) & (2,3) \\
(3,0) & (3,1) & (3,2) & (3,3)
\end{bmatrix}

**Column-Major Layout (Fortran-style):**
\begin{bmatrix}
(0,0) & (1,0) & (2,0) & (3,0) \\
(0,1) & (1,1) & (2,1) & (3,1) \\
(0,2) & (1,2) & (2,2) & (3,2) \\
(0,3) & (1,3) & (2,3) & (3,3)
\end{bmatrix}

### Identity Tensor

An identity tensor $I$ is a special case of a coordinate tensor that implements the identity mapping function:

**Definition:**
For a given shape $S = (s_1, s_2, ..., s_n)$, the identity tensor $I$ satisfies: $I(c) = c, \forall c \in \prod_{i=1}^n [0, s_i)$

**Properties:**
1. **Bijective Mapping**: The identity tensor establishes a one-to-one correspondence between coordinates.
2. **Layout Invariance**: The logical structure remains constant regardless of the underlying memory layout.
3. **Coordinate Preservation**: For any coordinate c, I(c) = c.


CuTe establishes an isomorphism between 1-D indices and N-D coordinates through lexicographical ordering. For a coordinate c = (c₁, c₂, ..., cₙ) in an identity tensor with shape S = (s₁, s₂, ..., sₙ):

**Linear Index Formula:**
$\text{idx} = c_1 + \sum_{i=2}^{n} \left(c_i \prod_{j=1}^{i-1} s_j\right)$

**Example:**
```python
# Create an identity tensor from a given shape
coord_tensor = make_identity_tensor(layout.shape())

# Access coordinate using linear index
coord = coord_tensor[linear_idx]  # Returns the N-D coordinate
```

This bidirectional mapping enables efficient conversion from linear indices to N-dimensional coordinates, facilitating tensor operations and memory access patterns.